# GEIGER-1911 · 4 — How long at each, and when to stop

Eight places to point the counter and a shape to cut in the brass. What is left is two
hundred hours, and the two criteria notebook 2 produced do not agree about how to spend
them: one wants everything at five degrees, the other wants everything at a hundred and
fifty.

Neither gets it, and the reason is that the wide-angle station's evidence is only worth
anything if the quantities that would explain it away have already been measured. That
turns the allocation into a lexicographic problem rather than an optimization: **buy every
constraint the argument will need, as cheaply as it can be bought, and spend the rest
where the evidence cannot be talked out of.**

| | decides | reaches into |
|---|---|---|
| **1** | how the hours split | `design.fisher_information`, `design.expected_posterior_sd` |
| **2** | how little the anchor can get away with | `design.power_from_se` |
| **3** | what bound comes back | `weight_of_evidence` |
| **4** | when to stop, and how weak a source would do | `design.obrien_fleming`, `design.monitor`, `design.operating_characteristics` |

In [ ]:
import sys

sys.path.insert(0, ".")

import math

import numpy as np
import plotly.graph_objects as go

import scattering as S
from axiom.design import (
    FisherInformation, LookSchedule, MonitoringPath, PowerResult, StoppingRule,
    expected_posterior_sd, fisher_information, information_fractions, monitor,
    obrien_fleming, operating_characteristics, power_from_se,
)

from axiom.display import enable, table

enable();  # every axiom result renders itself from here on

surface = S.surface()
VISIBLE = S.truth(math.log(0.30))


def score(stations):
    # Everything an allocation has to be judged on, from one Fisher matrix and two sums.
    angle = np.array([a for a, _, _ in stations], dtype=float)
    hours = np.array([h for _, h, _ in stations], dtype=float)
    foil = np.array([f for _, _, f in stations], dtype=float)
    aperture = np.where(foil == 0.0, S.OMEGA_MAX, S.widest_aperture(angle, S.HARD_CENTRE))
    info = fisher_information(surface, S.station(angle, aperture, hours * 3600.0, foil),
                              VISIBLE, 1.0, method="finite")
    assert isinstance(info, FisherInformation)
    sds = expected_posterior_sd(S.PRIOR_SDS, info)
    assert isinstance(sds, dict)
    inn = foil == 1.0
    seconds = hours[inn] * 3600.0
    believed = S.weight_of_evidence(
        S.rate_per_steradian(angle[inn], S.HARD_CENTRE) * aperture[inn] * seconds,
        S.rate_per_steradian(angle[inn], S.DIFFUSE) * aperture[inn] * seconds).sum()
    attacked = S.surviving_evidence(angle[inn], aperture[inn], seconds)[0].sum()
    return float(believed), float(attacked), sds

## 1 · How the hours split

Four allocations of the same two hundred hours, scored four ways: the evidence if the
model of the core is believed, the evidence that survives the core being attacked, and
whether the two quantities the attack turns on — the core's width and the background —
came out measured.

In [ ]:
CANDIDATES = {
    "all of it at 5 degrees": [(5.0, 200.0, 1.0)],
    "all of it at 150 degrees": [(150.0, 200.0, 1.0)],
    "the mid bank only": [(5.0, 60.0, 1.0), (10.0, 50.0, 1.0), (20.0, 50.0, 1.0),
                          (45.0, 40.0, 1.0)],
    "an even split over eight": [(a, 25.0, 1.0) for a in
                                 (1.0, 2.5, 5.0, 10.0, 20.0, 45.0, 90.0, 150.0)],
    "the plan": [(s.theta_deg, s.hours, s.foil) for s in S.PLAN],
}

rows = []
for label, stations in CANDIDATES.items():
    believed, attacked, sds = score(stations)
    rows.append([label, f"{believed:.4g}", f"{attacked:.4g}", f"{sds['log_w']:.4g}",
                 f"{sds['log_b']:.4g}"])
table(rows, headers=("campaign", "nats believed", "nats attacked", "sd log_w", "sd log_b"))

The two single-station allocations are each optimal on their own criterion and useless.

**Two hundred hours at five degrees** collects nineteen million nats of evidence and
leaves the core's width and the background at their prior standard deviations — so the
attack in notebook 2 goes through unopposed and better than 99.9 per cent of those nats
evaporate.

**Two hundred hours at a hundred and fifty degrees** loses nothing to the attack, because
there is nothing there to attack, and it also cannot say what its own background is to
better than a factor of eight. A million nats that rest on an unmeasured background are
not a million nats.

An **even split over the eight stations** is the first honest allocation in the table:
every nuisance measured, and two hundred thousand nats surviving. The plan beats it by a
factor of two and a half, and it does so by noticing something the even split does not —
that the hours in the middle of the range are, under attack, worth almost nothing, and
that moving them outward costs nothing anybody will miss.

It is worth seeing that as a rate per hour, because the ratio is not subtle.

In [ ]:
foil_in = [s for s in S.PLAN if s.foil]
angles = np.array([s.theta_deg for s in foil_in])
aperture = S.widest_aperture(angles, S.HARD_CENTRE)
per_hour_believed = S.weight_of_evidence(
    S.rate_per_steradian(angles, S.HARD_CENTRE) * aperture * 3600.0,
    S.rate_per_steradian(angles, S.DIFFUSE) * aperture * 3600.0)
per_hour_attacked, _ = S.surviving_evidence(angles, aperture, 3600.0)

total = (per_hour_attacked * np.array([s.hours for s in foil_in])).sum()
table(
    [
        [f"{st.theta_deg:.1f}", f"{st.hours:.0f}", f"{believed:,.0f}", f"{attacked:,.0f}",
         f"{attacked * st.hours / total:.1%}"]
        for st, believed, attacked in zip(foil_in, per_hour_believed, per_hour_attacked)
    ],
    headers=("station (deg)", "hours", "nats/h believed", "nats/h attacked", "share of total"),
)
print(f"\nAn hour at 150 degrees is worth {per_hour_attacked[-1] / per_hour_attacked[2]:,.0f} "
      f"hours at 5 degrees, once the opposition has spoken.")
print("So the mid bank is bought only down to what section 2 says it has to be, and")
print("everything left over goes to the witness.")

## 2 · How little the anchor can get away with

The mid bank and the anchor are not there for their evidence. They are there to take the
opposition's freedom away — specifically, to pin the width of the multiple-scattering core
so that "your core is wider than you think" becomes a checkable claim rather than a
rhetorical one.

That is a power question with a well-posed effect size. *Would this plan notice if the
core were ten per cent wider than fitted?* `expected_posterior_sd` gives the standard
error, `power_from_se` turns it into a probability, and the two together say how few hours
the anchor actually needs.

In [ ]:
TEN_PER_CENT = math.log(1.1)
rows = []
for hours in (0.5, 1.0, 2.0, 5.0, 10.0, 20.0):
    stations = [(1.0, hours, 1.0), (2.5, hours, 1.0)] + \
               [(s.theta_deg, s.hours, s.foil) for s in S.PLAN if s.role != "anchor"]
    _, _, sds = score(stations)
    result: PowerResult = power_from_se(TEN_PER_CENT, sds["log_w"])
    rows.append((hours, sds["log_w"], result.power))
table(
    [[f"{h:.1f}", f"{w:.4f}", f"{pw:.4f}"] for h, w, pw in rows],
    headers=("hours at each anchor", "sd of log core width", "power against a 10% error"),
)

print(f"\nAnything from about two hours is certain to catch it; the plan books five, which")
print(f"is a factor of two of margin on the one measurement the whole argument leans on.")

Two hours would do and the plan spends five, which is a cheap way to buy margin on the
one quantity the whole defence rests on. Ten of the two hundred hours secure the other
hundred and ninety.

## 3 · What bound comes back

The experiment does not return the size of the nucleus; notebook 1 showed it cannot. What
it returns is the largest positive charge that would have been noticed. That curve is
computable before anything is switched on: for each candidate radius, the evidence this
plan expects to accumulate against it.

In [ ]:
radii = np.geomspace(3.0e-14, 1.0e-12, 240)
plan_seconds = np.array([s.seconds for s in foil_in])
plan_data = S.station(angles, aperture, plan_seconds)
mu_hard = surface.counts(plan_data, S.HARD_CENTRE)
evidence = np.array([float(S.weight_of_evidence(
    mu_hard, surface.counts(plan_data, S.truth(S.lam_of(r)))).sum()) for r in radii])

fig = S.figure("The largest positive charge this plan would have noticed",
               "radius of the positive charge (fm)",
               "expected evidence against it (nats)", height=440)
fig.add_trace(go.Scatter(x=radii * 1e15, y=np.maximum(evidence, 1e-3),
                         line={"color": S.HARD_COLOR, "width": 3}, showlegend=False))
fig.add_hline(y=3.0, line={"dash": "dash", "color": S.SLIT_COLOR},
              annotation_text="a Bayes factor of 20")
fig.add_vline(x=S.D_CLOSEST / 2 * 1e15, line={"dash": "dot", "color": S.TRUTH_COLOR},
              annotation_text="D/2 — the beam cannot see past this")
fig.update_xaxes(type="log")
fig.update_yaxes(type="log", exponentformat="power")
fig.show()

crossed = np.where(evidence > 3.0)[0]
bound = radii[crossed[0]]
print(f"the plan reports:  R < {bound:.3g} m = {bound * 1e15:.1f} fm")
print(f"the beam's floor:        {S.D_CLOSEST / 2 * 1e15:.1f} fm")
print(f"Rutherford, 1911:  R < 3.4e-14 m = 34.0 fm")

**Thirty-two femtometres**, against a floor of twenty-nine point six that no amount of
counting can beat. The plan gets within nine per cent of the best this beam can do — which
also means there is nothing left to buy with time, and the next improvement has to be
bought with energy.

Rutherford published $3.4 \times 10^{-14}$ m in 1911. The agreement is not a coincidence;
it is the same arithmetic, and the reason his bound was so close to the floor is that a
charge just above $D/2$ only bites at the very back angles. Which is worth checking,
because it says which station is paying for the bound.

In [ ]:
def bound_from(subset):
    ang = np.array([s.theta_deg for s in subset], dtype=float)
    sec = np.array([s.seconds for s in subset], dtype=float)
    ap = S.widest_aperture(ang, S.HARD_CENTRE)
    data = S.station(ang, ap, sec)
    hard = surface.counts(data, S.HARD_CENTRE)
    ev = np.array([float(S.weight_of_evidence(
        hard, surface.counts(data, S.truth(S.lam_of(r)))).sum()) for r in radii])
    hit = np.where(ev > 3.0)[0]
    return radii[hit[0]] * 1e15 if hit.size else math.nan


print(f"the whole plan                    R < {bound_from(foil_in):.1f} fm")
print(f"without the 150 degree witness    R < {bound_from([s for s in foil_in if s.theta_deg != 150.0]):.1f} fm")
print(f"the anchor and bank alone         R < {bound_from([s for s in foil_in if s.theta_deg < 60.0]):.1f} fm")
print(f"the floor                             {S.D_CLOSEST / 2 * 1e15:.1f} fm")
print("\nThe witness stations are doing two jobs at once. They carry the evidence that")
print("cannot be argued with, and they are the only place a charge ball just above D/2")
print("shows itself at all -- so they buy most of the bound as well.")

## 4 · When to stop

A hundred and ten hours at a hundred and fifty degrees is the largest single commitment in
the plan, and there is no reason to make it blind. The station has a clean interim
statistic — counts against a background that the foil-out run has measured — and looking
at it repeatedly costs error rate unless the looks are budgeted for. `obrien_fleming`
budgets them.

In [ ]:
HOURS = (5.0, 15.0, 30.0, 60.0, 110.0)
looks = LookSchedule(labels=tuple(f"{h:g} h" for h in HOURS),
                     information=information_fractions(HOURS))
boundary = obrien_fleming(0.05, looks, side="upper", kind="efficacy")
rule = StoppingRule(name="the witness at 150 degrees", looks=looks, boundaries=(boundary,))

table(
    [
        [label, f"{frac:.2f}", f"{z:.3f}"]
        for label, frac, z in zip(looks.labels, looks.information, boundary.z)
    ],
    headers=("look", "information", "stop above z"),
)

signal = float(S.rate_per_steradian([150.0], S.HARD_CENTRE)[0] - S.BACKGROUND_DENSITY) * S.OMEGA_MAX
background = S.BACKGROUND_DENSITY * S.OMEGA_MAX
full = 110 * 3600.0
print(f"\nat 150 degrees: {signal * 3600:.0f} counts/hour of signal against "
      f"{background * 3600:.2f} of background")
print(f"over the full 110 hours: {signal * full:,.0f} against {background * full:,.0f}")

The question the rule is really being asked is not *will this station settle it* — at this
source strength the answer is yes before the first look. It is **how weak a source would
still settle it**, which is the question that decides whether an apparatus is worth
building. `operating_characteristics` answers it directly: scale the source down and read
off the probability of ever crossing.

In [ ]:
rows = []
for factor in (1.0, 1e-2, 1e-3, 3e-4, 2e-4, 1e-4):
    drift = signal * factor * full / math.sqrt((signal * factor + background) * full)
    oc = operating_characteristics(rule, drift=drift)
    rows.append([f"{factor:.0e}x", f"{drift:.4g}", f"{oc.crossings.stop_probability:.4f}",
                 f"{oc.expected_looks:.2f}", f"{oc.expected_information * 110:.1f}"])
table(rows, headers=("source", "drift at 110 h", "P(stop)", "looks used", "hours used"))

quiet = operating_characteristics(rule, drift=0.0)
print(f"\nif the diffuse atom is right (drift 0), the rule stops anyway with probability "
      f"{quiet.crossings.stop_probability:.3f}")
print("-- which is alpha, and is what budgeting the looks bought.")

**A source a thousand times weaker than this one still settles the question with
certainty**, in about forty of the hundred and ten hours; three thousand times weaker
settles it three times in four. That is the number that says whether the apparatus is
worth building, and it is roughly the situation Geiger and Marsden were actually in —
which is why their answer took months of nights at the microscope rather than an
afternoon.

Here is what one of those weaker campaigns looks like from the inside.

In [ ]:
rng = np.random.default_rng(11)
FACTOR = 1e-3
elapsed = np.array(HOURS) * 3600.0
expected = (signal * FACTOR + background) * elapsed
counts = np.cumsum(rng.poisson(np.diff(np.concatenate([[0.0], expected]))))
z = (counts - background * elapsed) / np.sqrt(np.maximum(counts, 1))

path: MonitoringPath = monitor(rule, list(z))
print(f"a source {FACTOR:.0e} times the modelled one, at 150 degrees:\n")
rows = []
for outcome, seen, expect in zip(path.looks, counts, background * elapsed):
    rows.append([outcome.label, seen, f"{expect:.1f}", f"{outcome.z:.2f}",
                 f"{boundary.z[outcome.look]:.3f}",
                 "STOP - excess" if outcome.decision != "continue" else ""])
table(rows, headers=("look", "counts", "background", "z", "boundary", ""))
print(f"\n{path.decision} at look {path.stopped_at + 1}, after "
      f"{HOURS[path.stopped_at]:g} of the 110 hours")

A hundred and nine flashes where the background accounts for sixty-five, and the rule
stops. It is not a large excess and it does not need to be: a hundred and fifty degrees is
far past the point where the diffuse atom's prediction went under the background, so the
excess has nowhere to come from.

The hours that would have gone to the last look are not saved — they go back into the
same station, because it was the bound in section 3 they were buying, not the decision.

## What notebook 4 established

* The two criteria from notebook 2 pick opposite allocations and both are useless alone.
  The resolution is **lexicographic**: buy the constraints, then spend the remainder where
  the evidence is unarguable. That puts **a hundred and ten of two hundred hours at a
  hundred and fifty degrees** — which looked absurd when the criterion was the believed
  model, and is forced once it is not.
* **Two hours at each anchor** would pin the core width well enough to catch a ten per
  cent error with certainty. The plan books five, and those ten hours are what make the
  other hundred and ninety mean anything.
* The plan reports **R < 32 fm** against a floor of 29.6 fm set by the beam energy. There
  is nothing more to buy with time. Rutherford published 34 fm.
* The witness at a hundred and fifty degrees settles the question at the **first look, five
  hours in**, and would still settle it with a source a thousand times weaker. The rest of
  its hundred and ten hours are buying the bound, not the decision.

Notebook 5 runs it.